# 1. Reading the data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

df = pd.read_csv('exams.csv')
df['total_score'] = df['math score'] + df['reading score'] + df['writing score']

# 2. Task type

I am trying to predict total_score => my task is regression.

# 3. Preprocessing

I will not normalize because there are no numerical columns in X

In [2]:
X = df.drop(['math score', 'reading score', 'writing score', 'total_score'], axis=1)
y = df['total_score']

categorical_features = X.columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(800, 17)
(200, 17)


# 4. Training

In [3]:
print("Linear Regression")
lin_reg = LinearRegression()
lin_reg.fit(X_train_processed, y_train)
y_pred_lin = lin_reg.predict(X_test_processed)

mae_lin = mean_absolute_error(y_test, y_pred_lin)
print(f"Mean Absolute Error: {mae_lin:.2f}")


print("Decision Tree Regressor (max_depth=10)")
tree_reg = DecisionTreeRegressor(max_depth=10, random_state=42)
tree_reg.fit(X_train_processed, y_train)
y_pred_tree = tree_reg.predict(X_test_processed)

mae_tree = mean_absolute_error(y_test, y_pred_tree)
print(f"Mean Absolute Error: {mae_tree:.2f}")


print("K-Nearest Neighbors Regressor (k=20)")
knn_reg = KNeighborsRegressor(n_neighbors=20)
knn_reg.fit(X_train_processed, y_train)
y_pred_knn = knn_reg.predict(X_test_processed)

mae_knn = mean_absolute_error(y_test, y_pred_knn)
print(f"Mean Absolute Error: {mae_knn:.2f}")


print("Random Forest Regressor")
forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)
forest_reg.fit(X_train_processed, y_train)
y_pred_forest = forest_reg.predict(X_test_processed)

mae_forest = mean_absolute_error(y_test, y_pred_forest)
print(f"Mean Absolute Error: {mae_forest:.2f}")

Linear Regression
Mean Absolute Error: 30.75
Decision Tree Regressor (max_depth=10)
Mean Absolute Error: 32.47
K-Nearest Neighbors Regressor (k=20)
Mean Absolute Error: 32.14
Random Forest Regressor
Mean Absolute Error: 32.27


I played around with max depth and k and found the best values for the lowest mean squared error. (it is still very high because you can't predict test scores based on what you ate etc)

# 5. Metrics

In [4]:
models = {
    "Linear Regression": lin_reg,
    "Decision Tree": tree_reg,
    "K-Nearest Neighbors": knn_reg,
    "Random Forest": forest_reg
}

for name, model in models.items():
    y_train_pred = model.predict(X_train_processed)
    
    y_test_pred = model.predict(X_test_processed)
    
    mae_train = mean_absolute_error(y_train, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    mape_train = mean_absolute_percentage_error(y_train, y_train_pred)
    
    mae_test = mean_absolute_error(y_test, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mape_test = mean_absolute_percentage_error(y_test, y_test_pred)
    
    print(f"{name}:")
    print(f"Train MAE: {mae_train:.2f} | Test MAE: {mae_test:.2f}")
    print(f"Train RMSE: {rmse_train:.2f} | Test RMSE: {rmse_test:.2f}")
    print(f"Train MAPE: {mape_train:.2%} | Test MAPE: {mape_test:.2%}\n")

Linear Regression:
Train MAE: 29.40 | Test MAE: 30.75
Train RMSE: 36.11 | Test RMSE: 37.79
Train MAPE: 16.03% | Test MAPE: 16.89%

Decision Tree:
Train MAE: 24.97 | Test MAE: 32.47
Train RMSE: 31.89 | Test RMSE: 40.40
Train MAPE: 13.63% | Test MAPE: 17.71%

K-Nearest Neighbors:
Train MAE: 29.61 | Test MAE: 32.14
Train RMSE: 36.41 | Test RMSE: 39.35
Train MAPE: 16.17% | Test MAPE: 17.63%

Random Forest:
Train MAE: 25.03 | Test MAE: 32.27
Train RMSE: 31.64 | Test RMSE: 40.29
Train MAPE: 13.65% | Test MAPE: 17.73%



# 6. Comparisons and concluions

The linear regression model was the best (MAE: 30.8)
There is overfitting in the decision tree and random forest because I selected too high max depth and k

All models are probably underfitting because the error is too large and because you can't really predict the test scores properly with the data provided.

Improving the model can be easily done by adding more features (grades for example)
